# EIA and Open-Meteo ingestion

Pull data into Polars, inspect fields and units, and build a small grid-demand and weather dataset.

Follow the [README](README.md) to set up uv and launch Jupyter. The Open-Meteo cells run without a key. For EIA, set `EIA_API_KEY` in your local `.env` and launch with `uv run --env-file .env jupyter lab data_ingestion.ipynb`.

In [ ]:
import logging
from pathlib import Path

import polars as pl
import requests

from API import EIAClient, OpenMeteoClient

# Uncomment both lines to see sanitized request parameters and page offsets.
# logging.basicConfig(level=logging.INFO)
# logging.getLogger("API").setLevel(logging.DEBUG)

session = requests.Session()
meteo = OpenMeteoClient(session)

latitude, longitude = 42.36, -71.09  # Cambridge, MA
start_date, end_date = "2025-07-15", "2025-07-17"

## 1. Geocoding

Search for places, then use `admin1` and `country_code` to distinguish cities with the same name. Coordinates can feed any weather or elevation query.

In [ ]:
places = meteo.get_geocoding("Cambridge", countryCode="US", count=10)
pl.from_dicts(
    places.get("results", []),
    schema={
        "name": pl.String,
        "admin1": pl.String,
        "country_code": pl.String,
        "latitude": pl.Float64,
        "longitude": pl.Float64,
        "timezone": pl.String,
    },
).head()

## 2. Weather forecast

`hourly` and `daily` hold column arrays. `current` holds a single record. The forecast starts on the current date and changes when rerun.

In [ ]:
forecast = meteo.get_weather_forecast(
    latitude,
    longitude,
    hourly=["temperature_2m", "wind_speed_10m"],
    daily=["temperature_2m_max"],
    current=["temperature_2m"],
    forecast_days=1,
    timezone="GMT",
)
print(forecast["hourly_units"])
display(pl.DataFrame(forecast["hourly"]).head())
display(pl.DataFrame(forecast["daily"]))
display(pl.DataFrame([forecast["current"]]))

## 3. Historical weather

Fetch a closed date window from the archive. Convert the GMT time strings into timezone-aware UTC timestamps for the EIA join.

In [ ]:
archive = meteo.get_historical_weather(
    latitude,
    longitude,
    start_date,
    end_date,
    hourly=["temperature_2m", "wind_speed_10m", "shortwave_radiation"],
    timezone="GMT",
)
weather = (
    pl.DataFrame(archive["hourly"])
    .rename({
        "time": "timestamp_utc",
        "temperature_2m": "temperature_2m_c",
        "wind_speed_10m": "wind_speed_10m_kmh",
        "shortwave_radiation": "shortwave_radiation_wm2",
    })
    .with_columns(
        pl.col("timestamp_utc").str.to_datetime("%Y-%m-%dT%H:%M", time_zone="UTC")
    )
)
print(archive["hourly_units"])
weather.head()

## 4. Air quality forecast

Pull modeled fine and coarse particulate concentrations. The response gives their units.

In [ ]:
air = meteo.get_air_quality(
    latitude, longitude, hourly=["pm2_5", "pm10"], forecast_days=1, timezone="GMT"
)
print(air["hourly_units"])
pl.DataFrame(air["hourly"]).head()

## 5. Marine forecast

Use a point offshore east of Massachusetts for wave conditions.

In [ ]:
marine = meteo.get_marine_forecast(
    42.35, -70.40, hourly=["wave_height", "wave_period"], forecast_days=1, timezone="GMT"
)
print(marine["hourly_units"])
pl.DataFrame(marine["hourly"]).head()

## 6. Elevation

Batch two locations in one request. Elevations are in meters and follow the input order.

In [ ]:
sites = pl.DataFrame({
    "site": ["Cambridge", "Mount Washington"],
    "latitude": [latitude, 44.27],
    "longitude": [longitude, -71.30],
})
elevation = meteo.get_elevation(sites["latitude"].to_list(), sites["longitude"].to_list())
sites.with_columns(pl.Series("elevation_m", elevation["elevation"]))

## 7. EIA route discovery and facets

Browse from the root to an electricity dataset. Dataset metadata gives the accepted frequency, data field, and filter names. The facet endpoint gives the filter values. EIA uses `ISNE` for ISO New England and `D` for actual demand.

In [ ]:
eia = EIAClient(session)
demand_route = "electricity/rto/region-data"

display(pl.from_dicts(eia.get_metadata()["routes"]).select("id", "name").head())
display(pl.from_dicts(eia.get_metadata("electricity/rto")["routes"]).select("id", "name"))

metadata = eia.get_metadata(demand_route)
print("Data fields", list(metadata["data"]))
print("Frequencies", [frequency["id"] for frequency in metadata["frequency"]])
display(pl.from_dicts(metadata["facets"]))
display(pl.from_dicts(eia.get_facet(demand_route, "type")["facets"]))

## 8. EIA hourly demand

EIA uses UTC hour-ending labels for this frequency. The small page size intentionally exercises pagination. Leave it at the default of 5,000 for larger pulls. Raw API values may arrive as strings, so convert the measure before analysis.

In [ ]:
period = {
    "frequency": "hourly",
    "start": f"{start_date}T00",
    "end": f"{end_date}T23",
}
demand_rows = eia.iter_data(
    demand_route,
    **period,
    facets={"respondent": "ISNE", "type": "D"},
    page_size=25,
)
demand = pl.from_dicts(demand_rows).with_columns(
    # EIA gives the hour without minutes, such as 2025-07-15T00.
    timestamp_utc=(pl.col("period") + ":00").str.to_datetime("%Y-%m-%dT%H:%M", time_zone="UTC"),
    demand_mwh=pl.col("value").cast(pl.Float64),
)
demand.select("timestamp_utc", "respondent", "demand_mwh", "value-units").head()

## 9. EIA generation by fuel

Switch datasets and use the fuel code to break ties when sorting. Pivot the long result into one column per fuel type. Missing fuel-hour rows stay missing.

In [ ]:
generation_rows = eia.iter_data(
    "electricity/rto/fuel-type-data",
    **period,
    facets={"respondent": "ISNE"},
    sort=("period", "fueltype"),
)
generation = pl.from_dicts(generation_rows).with_columns(
    timestamp_utc=(pl.col("period") + ":00").str.to_datetime("%Y-%m-%dT%H:%M", time_zone="UTC"),
    generation_mwh=pl.col("value").cast(pl.Float64),
)
generation.pivot(on="fueltype", index="timestamp_utc", values="generation_mwh").head()

## 10. Join and export

Pair each regional demand observation with weather at a Cambridge-area grid point using the same UTC hour-ending label. Temperature and wind are values at that time. Shortwave radiation is averaged over the preceding hour. The left join keeps demand hours even when weather is missing.

Rerunning this cell replaces the CSV.

In [ ]:
dataset = demand.select("timestamp_utc", "respondent", "demand_mwh").join(
    weather, on="timestamp_utc", how="left", validate="1:1", maintain_order="left"
)
output = Path("data/isone_demand_weather.csv")
output.parent.mkdir(exist_ok=True)
dataset.write_csv(output, datetime_format="%Y-%m-%d %H:%M:%S%:z")
print(f"Wrote {dataset.height} rows to {output}")
dataset.head()